# 02 · Publishing (Block B)

The provider decides what it offers and under which rules. Nothing leaves the provider in this notebook: it only configures its own control plane through its Management API.

## Concept: the EDC offer model

An offer is built from three separate objects:

```
Asset ◄─── assetsSelector ─── ContractDefinition ───► accessPolicy   (who may SEE the offer)
  │                                                └─► contractPolicy (what the AGREEMENT will require)
  └─ dataAddress: where the data really is (never shown to consumers)
```

| Object | Holds |
|---|---|
| **Asset** | `@id`, public `properties` (name, description, any DCAT metadata), and a **data address** |
| **Data address** | instructions for the provider's data plane: `type` (`HttpData`, `AzureStorage`…) plus type-specific fields (`baseUrl`, OAuth2 settings, `proxyQueryParams`…) |
| **Policy definition** | an ODRL policy |
| **Contract definition** | which assets (a selector over asset properties) are offered under which access and contract policies |

The separation means one policy can cover many assets, and one selector (for instance "every asset whose `dct:type` is `DATASET`") can publish assets that do not exist yet.

## Concept: ODRL policies and where they are evaluated

Policies use **ODRL** (Open Digital Rights Language):

```json
{"@type": "Set",
 "permission": [{"action": "use",
                 "constraint": {"leftOperand": "dse-policy:Membership", "operator": "eq", "rightOperand": "active"}}]}
```

A **permission** allows an **action** (`use`) if all its **constraints** hold. A constraint is `leftOperand operator rightOperand`.
What `leftOperand` *means* is decided by **policy functions** registered in the runtime. EDC evaluates the same policy at different moments, called **scopes**:

| Scope | When | Which policy |
|---|---|---|
| `catalog` | the consumer asks for the catalog | access policy |
| `contract.negotiation` | the consumer requests a contract | contract policy |
| `transfer.process` | a transfer starts | the agreement's policy |
| `policy.monitor` | periodically while a transfer runs | the agreement's policy |

The DSE policy functions available here:

| leftOperand | Checks | Scopes where it bites |
|---|---|---|
| `dse-policy:Membership` = `active` | an active MembershipCredential from the trusted authority | all |
| `dse-policy:GenericClaim.$.<Type>.<path>` | any claim of any credential type (JSON path) | negotiation (not catalog) |
| `dse-policy:RestrictedDiscoveryClaim.$.<Type>.<path>` | same, and **hides** the asset from catalogs when it fails | catalog **and** negotiation |
| `inForceDate` `lteq` `contractAgreement+20s` | agreement age | transfer, policy monitor |

## The four entries of this notebook

| Entry | Extra rule | What should happen |
|---|---|---|
| `data` | name = consumer | the consumer can contract it and read the data |
| `nobody` | name = nobody | visible, but the negotiation is rejected |
| `travel` | discovery restricted to domain travel | hidden from the consumer (nobody can hold `travel`) |
| `20s` | agreement valid 20 seconds | the transfer is stopped after 20 s |

In [1]:
import dataspace as ds

run_id = ds.recall("run_id", produced_by="00_environment.ipynb")
provider_api = ds.management_api("provider")

BACKEND_URL = "http://provider-backend:8080/api/provider/data"   # sample API that echoes its query parameters

## The three building blocks

One function per Management API call, so each body is easy to read:

| Function | Call | Returns |
|---|---|---|
| `create_asset` | `POST /v3/assets` | `IdResponse`: `{"@id": "...", "createdAt": ...}` |
| `create_policy` | `POST /v3/policydefinitions` | `IdResponse` with a generated policy id |
| `create_contract_definition` | `POST /v3/contractdefinitions` | `IdResponse` |

In [2]:
def create_asset(asset_id, description):
    asset = {
        "@context": ds.MANAGEMENT_CONTEXT,
        "@id": asset_id,
        "@type": "Asset",
        "properties": {"name": asset_id, "description": description, "contenttype": "application/json"},
        "dataAddress": {                      # where the provider's data plane will fetch the data
            "@type": "DataAddress",
            "type": "HttpData",
            "baseUrl": BACKEND_URL,
            "proxyQueryParams": "true",       # forward the consumer's ?query=params to the backend
        },
    }
    return ds.post(f"{provider_api}/assets", asset, expect={200}).json()


def create_policy(extra_constraints):
    constraints = [{"leftOperand": "dse-policy:Membership", "operator": "eq", "rightOperand": "active"}]
    constraints += extra_constraints
    policy_definition = {
        "@context": ds.MANAGEMENT_CONTEXT,
        "@type": "PolicyDefinition",
        "policy": {
            "@type": "Set",
            "permission": [{"action": "use", "constraint": constraint} for constraint in constraints],
        },
    }
    return ds.post(f"{provider_api}/policydefinitions", policy_definition, expect={200}).json()["@id"]


def create_contract_definition(asset_id, policy_id):
    contract_definition = {
        "@context": ds.EDC_CONTEXT,
        "@id": f"contract-{asset_id}",
        "@type": "ContractDefinition",
        "accessPolicyId": policy_id,          # who may see the offer
        "contractPolicyId": policy_id,        # what the agreement will require
        "assetsSelector": [                   # which assets this definition offers
            # the asset id property, written as its full JSON-LD IRI
            {"@type": "Criterion", "operandLeft": ds.EDC + "id", "operator": "=", "operandRight": asset_id}
        ],
    }
    ds.post(f"{provider_api}/contractdefinitions", contract_definition, expect={200})


def create_entry(asset_id, description, extra_constraints):
    create_asset(asset_id, description)
    policy_id = create_policy(extra_constraints)
    create_contract_definition(asset_id, policy_id)
    return asset_id, policy_id

## Entry `data`: only the participant named *consumer* may contract it

`GenericClaim.$.MembershipCredential.name eq consumer` reads the `name` claim of the consumer's MembershipCredential (created in notebook 01).

In [11]:
data_asset, data_policy_id = create_entry(
    asset_id=f"ging-data-{run_id}",
    description="readable by consumer",
    extra_constraints=[{"leftOperand": "dse-policy:GenericClaim.$.MembershipCredential.name", "operator": "eq", "rightOperand": "consumer"}],
)

POST   /provider/cp/management/v3/assets  ->  409


RuntimeError: expected status [200], got 409: [{"message":"Asset with ID ging-data-42930a already exists","type":"ObjectConflict","path":null,"invalidValue":null}]

### What was stored

**What:** `GET /v3/policydefinitions/<id>` and `GET /v3/assets/<id>`.

**Returns:** the objects as the control plane stores them. Note the `odrl:` prefixes in the response: the policy was expanded to full ODRL IRIs and compacted back.
The asset's data address is returned too (this is the provider's own API), but it never appears in the catalog consumers see.

In [4]:
ds.show(ds.get(f"{provider_api}/policydefinitions/{data_policy_id}").json()["policy"], max_chars=1200)

GET    /provider/cp/management/v3/policydefinitions/fb5465ab-b630-4f72-9461-f961958a7471  ->  200
{
  "@id": "5db6942e-d6c9-441c-a43a-48fc654a75c1",
  "@type": "odrl:Set",
  "odrl:permission": [
    {
      "odrl:action": {
        "@id": "odrl:use"
      },
      "odrl:constraint": {
        "odrl:leftOperand": {
          "@id": "dse-policy:Membership"
        },
        "odrl:operator": {
          "@id": "odrl:eq"
        },
        "odrl:rightOperand": "active"
      }
    },
    {
      "odrl:action": {
        "@id": "odrl:use"
      },
      "odrl:constraint": {
        "odrl:leftOperand": {
          "@id": "dse-policy:GenericClaim.$.MembershipCredential.name"
        },
        "odrl:operator": {
          "@id": "odrl:eq"
        },
        "odrl:rightOperand": "consumer"
      }
    }
  ],
  "odrl:prohibition": [],
  "odrl:obligation": []
}


In [5]:
stored_asset = ds.get(f"{provider_api}/assets/{data_asset}").json()
print("properties:  ", stored_asset["properties"])
print("dataAddress: ", stored_asset["dataAddress"])

GET    /provider/cp/management/v3/assets/ging-data-42930a  ->  200
properties:   {'createdAt': '2026-10-09T17:45:39.378Z', 'name': 'ging-data-42930a', 'description': 'readable by consumer', 'id': 'ging-data-42930a', 'contenttype': 'application/json'}
dataAddress:  {'@type': 'DataAddress', 'type': 'HttpData', 'proxyQueryParams': 'true', 'baseUrl': 'http://provider-backend:8080/api/provider/data'}


## Entry `nobody`: a rule no participant satisfies

In [6]:
nobody_asset, _ = create_entry(
    asset_id=f"ging-nobody-{run_id}",
    description="nobody qualifies",
    extra_constraints=[{"leftOperand": "dse-policy:GenericClaim.$.MembershipCredential.name", "operator": "eq", "rightOperand": "nobody"}],
)

POST   /provider/cp/management/v3/assets  ->  200
POST   /provider/cp/management/v3/policydefinitions  ->  200
POST   /provider/cp/management/v3/contractdefinitions  ->  200


## Entry `travel`: only visible to holders of a `travel` DomainCredential

In [7]:
travel_asset, _ = create_entry(
    asset_id=f"ging-travel-{run_id}",
    description="travel domain only",
    extra_constraints=[{"leftOperand": "dse-policy:RestrictedDiscoveryClaim.$.DomainCredential.domain", "operator": "eq", "rightOperand": "travel"}],
)

POST   /provider/cp/management/v3/assets  ->  200
POST   /provider/cp/management/v3/policydefinitions  ->  200
POST   /provider/cp/management/v3/contractdefinitions  ->  200


## Entry `20s`: the agreement is only valid for 20 seconds

`inForceDate` is a standard EDC constraint. `contractAgreement+20s` is relative to the moment the agreement is signed.

In [8]:
short_lived_asset, _ = create_entry(
    asset_id=f"ging-20s-{run_id}",
    description="agreement valid 20 s",
    extra_constraints=[{"leftOperand": "inForceDate", "operator": "lteq", "rightOperand": "contractAgreement+20s"}],
)

POST   /provider/cp/management/v3/assets  ->  200
POST   /provider/cp/management/v3/policydefinitions  ->  200
POST   /provider/cp/management/v3/contractdefinitions  ->  200


In [9]:
ds.remember(data_asset=data_asset, nobody_asset=nobody_asset, travel_asset=travel_asset, short_lived_asset=short_lived_asset)

   remembered data_asset = ging-data-42930a
   remembered nobody_asset = ging-nobody-42930a
   remembered travel_asset = ging-travel-42930a
   remembered short_lived_asset = ging-20s-42930a


## Querying the provider's assets

**What:** `POST /v3/assets/request` with a **QuerySpec**, EDC's generic query object:

| Field | Meaning |
|---|---|
| `filterExpression` | list of `{operandLeft, operator, operandRight}` criteria, AND-ed |
| `limit`, `offset` | pagination (default limit is small) |
| `sortField`, `sortOrder` | ordering |

**Returns:** a JSON array of assets.

In [10]:
query = {"@context": ds.EDC_CONTEXT, "@type": "QuerySpec", "limit": 500}
assets = ds.post(f"{provider_api}/assets/request", query).json()

print(f"{len(assets)} assets in total; created in this run:")
for asset in assets:
    if asset["@id"].endswith(run_id):
        print("  ", asset["@id"])

POST   /provider/cp/management/v3/assets/request  ->  200
52 assets in total; created in this run:
   ging-data-42930a
   ging-nobody-42930a
   ging-travel-42930a
   ging-20s-42930a


Next: **03_discovery.ipynb**